

### Завдання 1: Виклик LLM з базовим промптом

**Мета:** навчитися викликати LLM через LangChain зі звичайним текстовим промптом.

**Що потрібно зробити:**

1. Створіть промпт, який дозволяє отримати інформацію простою мовою на тему "Квантові обчислення". Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

2. Обмежте відповідь до 200 символів і пропишіть в промпті, аби відповідь була короткою (це зекономить вам час і гроші на згенеровані токени).

3. Встановіть своє значення температури на власний розсуд (тут немає правильного чи неправильного значення) і напишіть коментарем, чому ви обрали саме таке значення для цього завдання.

**Вибір моделі:** можна скористатись як моделлю з HuggingFace, так і ChatGPT будь-якої версії, яка вам до вподоби і пасує за прайсингом. В обох випадках потрібно імпортувати відповідний клас з LangChain для виклику LLM за API.

**Мова запитів:** промпти можна писати як українською, так і англійською — орієнтуйтесь на те, де і для чого ви хочете потім використовувати цей проєкт. У розв'язках промпти — українською.

---

**🔐 Як безпечно зберігати і підвантажувати API-ключі**

API-токен потрібно зчитувати з безпечного джерела, а **не хардкодити в ноутбуці**. Якщо хтось отримає доступ до вашого ключа, він буде витрачати токени за ваш рахунок, а вам це не треба :)

Є кілька способів. Перший ми використовували на лекції, ще два для розширення вашого розуміння, як ще це можна зробити і що шлях не лише один. Для виконання цього ДЗ можете використовувати будь-який спосіб підвантаження ключів у ноутбук.

**Спосіб 1: Файл `creds.json` (рекомендований)**

Створіть файл `creds.json` з вашими ключами, завантажте його в Google Colab під час роботи, але **не здавайте** цей файл у ДЗ і **не комітьте** в git.

```python
import json
with open("creds.json") as f:
    creds = json.load(f)
api_key = creds["HF_TOKEN"]
```

**Спосіб 2: Google Colab Secrets**

У лівій панелі Colab натисніть іконку 🔑 (Secrets) → "Add new secret" → введіть назву (наприклад, `HF_TOKEN`) та значення ключа → увімкніть тогл доступу для ноутбука.

```python
from google.colab import userdata
api_key = userdata.get("HF_TOKEN")
```

Зручно тим, що ключ зберігається в акаунті і доступний у всіх ваших ноутбуках. Мінус — при кожній новій сесії потрібно перевірити, що доступ увімкнено.

**Спосіб 3: Google AI Studio (для Gemini API)**

Якщо працюєте з моделями Google Gemini, отримати безкоштовний API-ключ можна в [Google AI Studio](https://aistudio.google.com/app/apikey): увійдіть з Google-акаунтом → натисніть "Get API key" → "Create API key". Далі використовуйте ключ через будь-який із способів вище.



In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import Runnable

# All api keys are in os.environ
import os

# use 0.1 to make responses quite conservative and less random
overall_temperature = 0.1

# create open ai chat model
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=overall_temperature,
    max_tokens=600, # limit output tokens
)

# define helper function
def query_llm(
    llm: Runnable[str, AIMessage],
    question: str,
    verbose: bool = True,
) -> AIMessage:
    """
    Send a prompt to a LangChain chat model and return the AIMessage response.

    Args:
        llm: Any LangChain chat model implementing `.invoke()` that returns `AIMessage`.
        question (str): The user prompt to send to the model.
        verbose (bool, optional): If True, prints the response content. Defaults to True.

    Returns:
        AIMessage:
            The complete response from the model.
    """
    response = llm.invoke(question)

    if verbose:
        print(response.content)

    return response


In [13]:
query_llm(llm, "Explain very shortly (under 500 characters if possible) and not too complicated what Quantum computing is. Include into response main definitions, key benefits, and current research in the field.");

Quantum computing is a type of computing that uses quantum bits (qubits) instead of classical bits. Qubits can exist in multiple states simultaneously, allowing quantum computers to process vast amounts of data more efficiently. Key benefits include solving complex problems faster, such as optimization, cryptography, and drug discovery. Current research focuses on improving qubit stability, error correction, and developing practical applications across various industries, including finance, healthcare, and materials science.


### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [14]:
prompt = PromptTemplate(
    input_variables=["topic"],
    template="Explain very shortly (under 500 characters if possible) and not too complicated what {topic} is. Include into response main definitions, key benefits, and current research in the field.",
)

print(prompt.format(topic="donuts"))

Explain very shortly (under 500 characters if possible) and not too complicated what donuts is. Include into response main definitions, key benefits, and current research in the field.


In [16]:
query_llm(llm, prompt.format(topic="Bayesian methods in machine learning"));

Bayesian methods in machine learning use Bayes' theorem to update the probability of a hypothesis as more evidence becomes available. Key definitions include prior (initial belief), likelihood (evidence), and posterior (updated belief). Benefits include handling uncertainty, incorporating prior knowledge, and providing probabilistic interpretations of predictions. Current research focuses on improving computational efficiency, developing scalable algorithms, and applying Bayesian methods to deep learning, reinforcement learning, and causal inference.


In [17]:
query_llm(llm, prompt.format(topic="Transformers in machine learning"));

Transformers are a type of neural network architecture designed for processing sequential data, primarily used in natural language processing (NLP). They rely on self-attention mechanisms to weigh the importance of different words in a sentence, allowing for better context understanding. Key benefits include parallel processing, improved performance on long-range dependencies, and scalability. Current research focuses on enhancing efficiency, reducing model size, and applying Transformers to various domains like computer vision and audio processing.


In [18]:
# use chain this time
chain = prompt | llm | StrOutputParser()

result = chain.invoke({"topic": "Explainable AI"})
print(result)

Explainable AI (XAI) refers to methods and techniques that make the outputs of AI systems understandable to humans. Key definitions include interpretability (how well a human can understand the model's decisions) and transparency (how clear the model's processes are). Benefits include increased trust, accountability, and compliance with regulations. Current research focuses on developing algorithms that provide insights into model behavior, creating visualizations for better understanding, and ensuring fairness and bias mitigation in AI systems.




### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [ ]:
!pip install -q langchain_community duckduckgo_search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 35.8 MB/s eta 0:00:00


In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Obama's first name?")

"2 of 2. Barack Obama: timeline Key events in the life of Barack Obama. Barack Obama (born August 4, 1961, Honolulu, Hawaii, U.S.) is the 44th president of the United States (2009-17) and the first African American to hold the office. Before winning the presidency, Obama represented Illinois in the U.S. Senate (2005-08). Since the office was established in 1789, 45 men have served in 46 presidencies. The first president, George Washington, won a unanimous vote of the Electoral College. [4] Grover Cleveland served two non-consecutive terms and is therefore counted as the 22nd and 24th president of the United States, giving rise to the discrepancy between the ... Here is a list of the presidents and vice presidents of the United States along with their parties and dates in office. ... Chester A Arthur: Twenty-First President of the United States. 10 Interesting Facts About James Buchanan. Martin Van Buren - Eighth President of the United States. Quotes From Harry S. Truman. Table of Cont



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?
